In [1]:
!pip install -q llama-index llama-index-embeddings-ollama llama-index-llms-ollama chromadb llama-index-vector-stores-chroma

print("✅ Required packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 659.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

In [2]:
from pathlib import Path
import time

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

print("✅ LlamaIndex libraries imported successfully!")

✅ LlamaIndex libraries imported successfully!


In [3]:
docs_folder = Path("w7d3_documents")
docs_folder.mkdir(exist_ok=True)

text_documents = {
    "python.txt": """
Python is a high-level programming language.
Python is widely used for web development, automation, data science, artificial intelligence, and machine learning.
Python has simple and readable syntax.
""",

    "machine_learning.txt": """
Machine learning is a branch of artificial intelligence.
It allows computers to learn patterns from data and make predictions.
Common machine learning tasks include classification and regression.
""",

    "database.txt": """
A database is an organized collection of data.
SQL databases store structured information using tables, rows, and columns.
SQL is commonly used to create, read, update, and delete data.
""",

    "rag.txt": """
Retrieval Augmented Generation, or RAG, combines information retrieval with a language model.
RAG retrieves relevant documents first and provides them to the language model as context.
This helps the model generate answers based on external information.
""",

    "llamaindex.txt": """
LlamaIndex is a framework for building applications using large language models.
It helps developers connect language models with external data.
LlamaIndex can index documents and provide query engines for retrieving relevant information.
"""
}

for filename, content in text_documents.items():
    file_path = docs_folder / filename
    file_path.write_text(content.strip(), encoding="utf-8")

print("✅ 5 text documents created!")

for file in sorted(docs_folder.glob("*.txt")):
    print("📄", file)

✅ 5 text documents created!
📄 w7d3_documents/database.txt
📄 w7d3_documents/llamaindex.txt
📄 w7d3_documents/machine_learning.txt
📄 w7d3_documents/python.txt
📄 w7d3_documents/rag.txt


In [5]:
# Install zstd required by Ollama
!apt-get update -qq
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("✅ Ollama installation completed!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (400 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Cleaning up old version at /us

In [6]:
import subprocess
import time
import requests

# Start Ollama server
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Check server
response = requests.get("http://127.0.0.1:11434/api/tags")

print("✅ Ollama server started!")
print("Status code:", response.status_code)

✅ Ollama server started!
Status code: 200


In [7]:
!ollama pull llama3.2:3b

print("✅ llama3.2:3b model downloaded!")


✅ llama3.2:3b model downloaded!


In [8]:
!ollama list

NAME           ID              SIZE      MODIFIED       
llama3.2:3b    a80c4f17acd5    2.0 GB    32 seconds ago    


In [9]:
llm = Ollama(
    model="llama3.2:3b",
    request_timeout=120.0
)

print("✅ LlamaIndex connected to Ollama LLM!")
print("Model: llama3.2:3b")

✅ LlamaIndex connected to Ollama LLM!
Model: llama3.2:3b


In [10]:
embed_model = OllamaEmbedding(
    model_name="llama3.2:3b",
    base_url="http://127.0.0.1:11434"
)

print("✅ Ollama embedding model configured!")
print("Model: llama3.2:3b")

✅ Ollama embedding model configured!
Model: llama3.2:3b


In [11]:
!ollama pull nomic-embed-text

print("✅ Ollama embedding model downloaded!")


✅ Ollama embedding model downloaded!


In [12]:
embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://127.0.0.1:11434"
)

print("✅ Ollama embedding model configured!")
print("Embedding model: nomic-embed-text")

✅ Ollama embedding model configured!
Embedding model: nomic-embed-text


In [13]:
documents = SimpleDirectoryReader(
    input_dir=str(docs_folder)
).load_data()

print("✅ Documents loaded successfully!")
print("Total documents:", len(documents))

for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc.metadata.get('file_name', 'unknown')}")

✅ Documents loaded successfully!
Total documents: 5
1. database.txt
2. llamaindex.txt
3. machine_learning.txt
4. python.txt
5. rag.txt


In [14]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

print("✅ LlamaIndex model settings configured!")
print("LLM: llama3.2:3b")
print("Embedding: nomic-embed-text")

✅ LlamaIndex model settings configured!
LLM: llama3.2:3b
Embedding: nomic-embed-text


In [15]:
index = VectorStoreIndex.from_documents(
    documents
)

print("✅ LlamaIndex VectorStoreIndex created!")
print("Documents indexed:", len(documents))

✅ LlamaIndex VectorStoreIndex created!
Documents indexed: 5


In [16]:
query_engine = index.as_query_engine(
    similarity_top_k=2
)

print("✅ LlamaIndex QueryEngine created!")
print("Retrieval: Top 2 relevant document chunks")

✅ LlamaIndex QueryEngine created!
Retrieval: Top 2 relevant document chunks


In [22]:
import subprocess
import time
import requests

# Stop old Ollama processes
!pkill -f "ollama serve" || true

time.sleep(2)

# Start Ollama again
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

response = requests.get("http://127.0.0.1:11434/api/tags")

print("✅ Ollama restarted!")
print("Status code:", response.status_code)
!ollama pull llama3.2:1b

print("✅ Smaller LLM downloaded!")
llm = Ollama(
    model="llama3.2:1b",
    request_timeout=120.0
)

Settings.llm = llm

print("✅ LlamaIndex now uses llama3.2:1b")
query_engine = index.as_query_engine(
    similarity_top_k=1
)

print("✅ Query engine recreated!")
test_query = "What is Python used for?"

response = query_engine.query(test_query)

print("Question:", test_query)
print("\nAnswer:")
print(response)

print("\n📚 Source documents:")
for source_node in response.source_nodes:
    print("-", source_node.metadata.get("file_name", "unknown"))

^C
✅ Ollama restarted!
Status code: 200

✅ Smaller LLM downloaded!
✅ LlamaIndex now uses llama3.2:1b
✅ Query engine recreated!
Question: What is Python used for?

Answer:
Python is widely used for a variety of applications, including:

- Data analysis and scientific computing
- Web development and data visualization
- Automation and scripting
- Machine learning and artificial intelligence
- Mobile and desktop application development

📚 Source documents:
- python.txt


In [23]:
evaluation_queries = [
    "What is Python used for?",
    "What is machine learning?",
    "How does machine learning make predictions?",
    "What is a database?",
    "How does an SQL database organize data?",
    "What is RAG?",
    "How does RAG provide context to a language model?",
    "What is LlamaIndex?",
    "What can LlamaIndex help developers build?",
    "What combines information retrieval with a language model?"
]

expected_sources = [
    "python.txt",
    "machine_learning.txt",
    "machine_learning.txt",
    "database.txt",
    "database.txt",
    "rag.txt",
    "rag.txt",
    "llamaindex.txt",
    "llamaindex.txt",
    "rag.txt"
]

print("✅ 10 evaluation queries created!")

for i, query in enumerate(evaluation_queries, 1):
    print(f"Q{i}. {query}")

✅ 10 evaluation queries created!
Q1. What is Python used for?
Q2. What is machine learning?
Q3. How does machine learning make predictions?
Q4. What is a database?
Q5. How does an SQL database organize data?
Q6. What is RAG?
Q7. How does RAG provide context to a language model?
Q8. What is LlamaIndex?
Q9. What can LlamaIndex help developers build?
Q10. What combines information retrieval with a language model?


In [24]:
query_results = []

for i, query in enumerate(evaluation_queries, 1):
    start_time = time.perf_counter()

    response = query_engine.query(query)

    end_time = time.perf_counter()
    latency = end_time - start_time

    source_files = list({
        node.metadata.get("file_name", "unknown")
        for node in response.source_nodes
    })

    query_results.append({
        "question": query,
        "answer": str(response),
        "sources": source_files,
        "latency": latency
    })

    print(f"Q{i}. {query}")
    print(f"Answer: {response}")
    print(f"Sources: {source_files}")
    print(f"Latency: {latency:.2f} seconds")
    print("-" * 70)

print("✅ All 10 queries completed!")

Q1. What is Python used for?
Answer: Python is used for a variety of tasks, including:

- Developing and testing scripts and programs
- Automating repetitive tasks and workflows
- Building data analysis and visualization tools
- Creating interactive and dynamic web applications
- Running scientific simulations and modeling
- Creating games and interactive applications
Sources: ['python.txt']
Latency: 6.46 seconds
----------------------------------------------------------------------
Q2. What is machine learning?
Answer: Machine learning is a subset of artificial intelligence that enables computers to identify patterns and make predictions or decisions based on data.
Sources: ['machine_learning.txt']
Latency: 4.60 seconds
----------------------------------------------------------------------
Q3. How does machine learning make predictions?
Answer: Machine learning algorithms analyze patterns in data, often represented as mathematical equations or complex models, to make predictions or de

In [25]:
latencies = [result["latency"] for result in query_results]

average_latency = sum(latencies) / len(latencies)

print("⏱️ Query Latency")
print("=" * 50)

for i, latency in enumerate(latencies, 1):
    print(f"Q{i}: {latency:.2f} seconds")

print("=" * 50)
print(f"Average latency: {average_latency:.2f} seconds")
print("✅ Latency measurement completed!")

⏱️ Query Latency
Q1: 6.46 seconds
Q2: 4.60 seconds
Q3: 3.40 seconds
Q4: 4.97 seconds
Q5: 3.99 seconds
Q6: 7.56 seconds
Q7: 6.49 seconds
Q8: 3.86 seconds
Q9: 4.25 seconds
Q10: 7.36 seconds
Average latency: 5.29 seconds
✅ Latency measurement completed!


In [26]:
import chromadb

from llama_index.vector_stores.chroma import ChromaVectorStore

# Create a persistent ChromaDB client
chroma_client = chromadb.PersistentClient(
    path="./w7d3_chroma_db"
)

# Create a collection
chroma_collection = chroma_client.get_or_create_collection(
    name="w7d3_documents"
)

# Connect ChromaDB collection to LlamaIndex
chroma_vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

print("✅ ChromaDB vector store created!")
print("Collection:", chroma_collection.name)

✅ ChromaDB vector store created!
Collection: w7d3_documents


In [27]:
from llama_index.core import StorageContext

chroma_storage_context = StorageContext.from_defaults(
    vector_store=chroma_vector_store
)

chroma_index = VectorStoreIndex.from_documents(
    documents,
    storage_context=chroma_storage_context
)

print("✅ LlamaIndex connected to ChromaDB!")
print("Documents indexed in ChromaDB:", len(documents))

✅ LlamaIndex connected to ChromaDB!
Documents indexed in ChromaDB: 5


In [28]:
chroma_query_engine = chroma_index.as_query_engine(
    similarity_top_k=1
)

print("✅ ChromaDB QueryEngine created!")
print("Retrieval: Top 1 relevant document")

✅ ChromaDB QueryEngine created!
Retrieval: Top 1 relevant document


In [29]:
test_query = "What is RAG?"

start_time = time.time()

response = chroma_query_engine.query(test_query)

latency = time.time() - start_time

print("Question:", test_query)

print("\nAnswer:")
print(response)

print("\n📚 Source documents:")
for source_node in response.source_nodes:
    print("-", source_node.metadata.get("file_name", "unknown"))

print(f"\n⏱️ Latency: {latency:.2f} seconds")

Question: What is RAG?

Answer:
RAG, or Retrieval Augmented Generation, is a technology that combines information retrieval with a language model to generate human-like answers.

📚 Source documents:
- rag.txt

⏱️ Latency: 3.55 seconds


In [30]:
chroma_query_results = []

for i, question in enumerate(evaluation_queries, 1):
    start_time = time.time()

    response = chroma_query_engine.query(question)

    latency = time.time() - start_time

    sources = [
        node.metadata.get("file_name", "unknown")
        for node in response.source_nodes
    ]

    result = {
        "question": question,
        "answer": str(response),
        "sources": sources,
        "latency": latency
    }

    chroma_query_results.append(result)

    print(f"\n{'='*60}")
    print(f"Query {i}: {question}")
    print(f"Answer: {response}")
    print(f"Sources: {sources}")
    print(f"Latency: {latency:.2f} seconds")

print("\n✅ All 10 ChromaDB queries completed!")


Query 1: What is Python used for?
Answer: Python is used for a variety of purposes, including building and automating tasks, as well as creating complex and intricate systems.
Sources: ['python.txt']
Latency: 5.03 seconds

Query 2: What is machine learning?
Answer: Machine learning is a subset of artificial intelligence that enables computers to learn and improve their performance on a task without being explicitly programmed.
Sources: ['machine_learning.txt']
Latency: 4.89 seconds

Query 3: How does machine learning make predictions?
Answer: Machine learning makes predictions by analyzing patterns in data, often using complex algorithms and statistical models.
Sources: ['machine_learning.txt']
Latency: 2.70 seconds

Query 4: What is a database?
Answer: A database is a complex and organized system that allows multiple entities, such as users, orders, or products, to be linked together with precise and controlled access.
Sources: ['database.txt']
Latency: 5.50 seconds

Query 5: How doe

In [31]:
correct_count = 0

print("📊 ChromaDB Source Verification")
print("=" * 60)

for i, result in enumerate(chroma_query_results):
    expected = expected_sources[i]
    actual_sources = result["sources"]

    is_correct = expected in actual_sources

    if is_correct:
        correct_count += 1

    print(f"\n{i+1}. {result['question']}")
    print(f"Expected: {expected}")
    print(f"Retrieved: {actual_sources}")
    print(f"Correct: {'✅ Yes' if is_correct else '❌ No'}")

chroma_accuracy = correct_count / len(evaluation_queries) * 100

print("\n" + "=" * 60)
print(f"✅ ChromaDB Source Accuracy: {chroma_accuracy:.1f}%")
print(f"Correct: {correct_count}/{len(evaluation_queries)}")

📊 ChromaDB Source Verification

1. What is Python used for?
Expected: python.txt
Retrieved: ['python.txt']
Correct: ✅ Yes

2. What is machine learning?
Expected: machine_learning.txt
Retrieved: ['machine_learning.txt']
Correct: ✅ Yes

3. How does machine learning make predictions?
Expected: machine_learning.txt
Retrieved: ['machine_learning.txt']
Correct: ✅ Yes

4. What is a database?
Expected: database.txt
Retrieved: ['database.txt']
Correct: ✅ Yes

5. How does an SQL database organize data?
Expected: database.txt
Retrieved: ['database.txt']
Correct: ✅ Yes

6. What is RAG?
Expected: rag.txt
Retrieved: ['rag.txt']
Correct: ✅ Yes

7. How does RAG provide context to a language model?
Expected: rag.txt
Retrieved: ['rag.txt']
Correct: ✅ Yes

8. What is LlamaIndex?
Expected: llamaindex.txt
Retrieved: ['llamaindex.txt']
Correct: ✅ Yes

9. What can LlamaIndex help developers build?
Expected: llamaindex.txt
Retrieved: ['llamaindex.txt']
Correct: ✅ Yes

10. What combines information retrieval w

In [32]:
chroma_latencies = [
    result["latency"]
    for result in chroma_query_results
]

chroma_average_latency = sum(chroma_latencies) / len(chroma_latencies)

print("⏱️ ChromaDB Latency Results")
print("=" * 50)

print(f"Average latency: {chroma_average_latency:.2f} seconds")
print(f"Fastest query: {min(chroma_latencies):.2f} seconds")
print(f"Slowest query: {max(chroma_latencies):.2f} seconds")

⏱️ ChromaDB Latency Results
Average latency: 5.26 seconds
Fastest query: 2.70 seconds
Slowest query: 9.73 seconds


In [34]:
# Calculate original LlamaIndex source accuracy

original_correct_count = 0

for i, result in enumerate(query_results):
    expected = expected_sources[i]
    actual_sources = result["sources"]

    if expected in actual_sources:
        original_correct_count += 1

source_accuracy = (
    original_correct_count / len(evaluation_queries) * 100
)

print("✅ Original LlamaIndex source accuracy calculated!")
print(f"Correct: {original_correct_count}/{len(evaluation_queries)}")
print(f"Accuracy: {source_accuracy:.1f}%")

✅ Original LlamaIndex source accuracy calculated!
Correct: 10/10
Accuracy: 100.0%


In [35]:
print("📊 LlamaIndex vs ChromaDB Comparison")
print("=" * 60)

print(f"Original LlamaIndex source accuracy : {source_accuracy:.1f}%")
print(f"ChromaDB source accuracy            : {chroma_accuracy:.1f}%")

print()

print(f"Original average latency : {average_latency:.2f} seconds")
print(f"ChromaDB average latency: {chroma_average_latency:.2f} seconds")

print()

latency_difference = chroma_average_latency - average_latency

print(f"Latency difference: {latency_difference:.2f} seconds")

print("\n✅ Comparison completed successfully!")

📊 LlamaIndex vs ChromaDB Comparison
Original LlamaIndex source accuracy : 100.0%
ChromaDB source accuracy            : 100.0%

Original average latency : 5.29 seconds
ChromaDB average latency: 5.26 seconds

Latency difference: -0.04 seconds

✅ Comparison completed successfully!


In [36]:
print("📝 W7D3 SELF-REVIEW")
print("=" * 60)

print("""
1. What did I build?
I built a LlamaIndex document indexing and querying system.
I indexed text documents using Ollama embeddings and created a QueryEngine.
I also connected LlamaIndex with ChromaDB as the vector store.

2. What design decisions did I make?
I used nomic-embed-text for embeddings and llama3.2:1b
for lightweight local LLM responses.
I used top-1 retrieval to keep the experiment simple and efficient.
I tested the same 10 questions before and after connecting ChromaDB.

3. What was the hardest part?
The main challenge was running the local Ollama model within
the available Colab memory. The 3B model was terminated,
so I switched the LLM to llama3.2:1b.

4. What did I learn?
I learned how LlamaIndex creates document indexes and query engines,
how Ollama can provide local embeddings and LLM responses,
and how ChromaDB can be used as a persistent vector store.

5. What would I improve?
I would use a larger and more reliable document collection,
better evaluation metrics, and experiment with different embedding
models and retrieval settings.
""")

print("✅ Self-review completed!")

📝 W7D3 SELF-REVIEW

1. What did I build?
I built a LlamaIndex document indexing and querying system.
I indexed text documents using Ollama embeddings and created a QueryEngine.
I also connected LlamaIndex with ChromaDB as the vector store.

2. What design decisions did I make?
I used nomic-embed-text for embeddings and llama3.2:1b
for lightweight local LLM responses.
I used top-1 retrieval to keep the experiment simple and efficient.
I tested the same 10 questions before and after connecting ChromaDB.

3. What was the hardest part?
The main challenge was running the local Ollama model within
the available Colab memory. The 3B model was terminated,
so I switched the LLM to llama3.2:1b.

4. What did I learn?
I learned how LlamaIndex creates document indexes and query engines,
how Ollama can provide local embeddings and LLM responses,
and how ChromaDB can be used as a persistent vector store.

5. What would I improve?
I would use a larger and more reliable document collection,
better eval